In [ ]:
#PL-LDGAN (A HYBRID LDM-GAN TRAINING)
# ============================================================
# # Encoder → 64x64x256 → UNet Diffusion → Decoder → PatchGAN
# AE + KL + Diffusion + GAN
# ============================================================

import os, torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import save_image
from PIL import Image
import torch.nn.functional as F
from tqdm import tqdm

# ================= CONFIG =================
IMG_SIZE = 256
BATCH = 16
TIMESTEPS = 500
LR = 2e-4
DATA_PATH = "/content/drive/MyDrive/POTATODATASET/Early_blight"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ================= DATA =================
class LeafDataset(Dataset):
    def __init__(self, root):
        self.paths = [os.path.join(root,f) for f in os.listdir(root)]
        self.tf = T.Compose([
            T.Resize((IMG_SIZE,IMG_SIZE)),
            T.ToTensor(),
            T.Normalize([0.5]*3,[0.5]*3)
        ])
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img)

loader = DataLoader(LeafDataset(DATA_PATH), batch_size=BATCH, shuffle=True)

# ================= ENCODER =================
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,4,2,1), nn.ReLU(),     # 256→128
            nn.Conv2d(64,128,4,2,1), nn.ReLU(),   # 128→64
            nn.Conv2d(128,256,3,1,1), nn.ReLU()   # keep 64
        )
    def forward(self,x): return self.net(x)      # [B,256,64,64]

# ================= DECODER =================
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(256,128,4,2,1), nn.ReLU(), # 64→128
            nn.ConvTranspose2d(128,64,4,2,1), nn.ReLU(),  # 128→256
            nn.Conv2d(64,3,3,1,1),
            nn.Tanh()
        )
    def forward(self,z): return self.net(z)

# ================= UNET DIFFUSION =================
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(c,c,3,1,1), nn.ReLU(),
            nn.Conv2d(c,c,3,1,1), nn.ReLU()
        )
    def forward(self,x): return self.b(x)

class LatentUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(TIMESTEPS,256)
        self.d1 = Block(256)
        self.d2 = Block(256)
        self.u1 = Block(256)
        self.out = nn.Conv2d(256,256,3,1,1)

    def forward(self,z,t):
        e = self.embed(t).view(-1,256,1,1)
        z = z + e
        d1 = self.d1(z)
        d2 = self.d2(d1)
        u1 = self.u1(d2 + d1)
        return self.out(u1)

# ================= PATCHGAN =================
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,4,2,1), nn.LeakyReLU(0.2),
            nn.Conv2d(32,64,4,2,1), nn.LeakyReLU(0.2),
            nn.Conv2d(64,128,4,2,1), nn.LeakyReLU(0.2),
            nn.Conv2d(128,1,4,1,0)
        )
    def forward(self,x): return self.net(x)

# ================= INIT =================
E, G, UNet, D = Encoder().to(device), Decoder().to(device), LatentUNet().to(device), Discriminator().to(device)

opt_G = optim.Adam(list(E.parameters())+list(G.parameters())+list(UNet.parameters()), lr=LR)
opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5,0.999))
bce = nn.BCEWithLogitsLoss()

# ================= DIFFUSION PARAMS =================
betas = torch.linspace(1e-4,0.02,TIMESTEPS).to(device)
alphas = 1 - betas
alpha_hat = torch.cumprod(alphas,0)

def kl_loss(z): return torch.mean(z**2)

# ================= TRAIN =================
print("🚀 Training TRUE Hybrid LDM-GAN")

for epoch in range(300):
    for imgs in tqdm(loader):
        imgs = imgs.to(device)

        # -------- Generator --------
        z = E(imgs)
        recon = G(z)

        ae = F.mse_loss(recon, imgs)
        kl = 0.001*kl_loss(z)

        t = torch.randint(0,TIMESTEPS,(z.size(0),),device=device)
        noise = torch.randn_like(z)
        a = alpha_hat[t].view(-1,1,1,1)

        z_noisy = torch.sqrt(a)*z + torch.sqrt(1-a)*noise
        pred_noise = UNet(z_noisy,t)
        diff = F.mse_loss(pred_noise, noise)

        gan = bce(D(recon), torch.ones_like(D(recon)))

        g_loss = ae + diff + kl + 0.01*gan
        opt_G.zero_grad(); g_loss.backward(); opt_G.step()

        # -------- Discriminator --------
        d_loss = 0.5*(bce(D(imgs),torch.ones_like(D(imgs))) +
                      bce(D(recon.detach()),torch.zeros_like(D(recon))))
        opt_D.zero_grad(); d_loss.backward(); opt_D.step()

    print(f"Epoch {epoch+1} | AE {ae:.3f} Diff {diff:.3f} GAN {gan:.3f}")

    if (epoch+1)%20==0:
        save_image((recon[:8]+1)/2, f"sample_{epoch+1}.png", nrow=4)


In [ ]:
#PL-LDGAN (A HYBRID LDM-GAN TRAINING)
# ============================================================
# # Encoder → 64x64x256 → UNet Diffusion → Decoder → PatchGAN
# AE + KL + Diffusion + GAN
# ============================================================

import os, torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import save_image
from PIL import Image
import torch.nn.functional as F
from tqdm import tqdm

# ================= CONFIG =================
IMG_SIZE = 256
BATCH = 16
TIMESTEPS = 500
LR = 2e-4
DATA_PATH = "/content/drive/MyDrive/POTATODATASET/Late_blight"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ================= DATA =================
class LeafDataset(Dataset):
    def __init__(self, root):
        self.paths = [os.path.join(root,f) for f in os.listdir(root)]
        self.tf = T.Compose([
            T.Resize((IMG_SIZE,IMG_SIZE)),
            T.ToTensor(),
            T.Normalize([0.5]*3,[0.5]*3)
        ])
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img)

loader = DataLoader(LeafDataset(DATA_PATH), batch_size=BATCH, shuffle=True)

# ================= ENCODER =================
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,4,2,1), nn.ReLU(),     # 256→128
            nn.Conv2d(64,128,4,2,1), nn.ReLU(),   # 128→64
            nn.Conv2d(128,256,3,1,1), nn.ReLU()   # keep 64
        )
    def forward(self,x): return self.net(x)      # [B,256,64,64]

# ================= DECODER =================
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(256,128,4,2,1), nn.ReLU(), # 64→128
            nn.ConvTranspose2d(128,64,4,2,1), nn.ReLU(),  # 128→256
            nn.Conv2d(64,3,3,1,1),
            nn.Tanh()
        )
    def forward(self,z): return self.net(z)

# ================= UNET DIFFUSION =================
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(c,c,3,1,1), nn.ReLU(),
            nn.Conv2d(c,c,3,1,1), nn.ReLU()
        )
    def forward(self,x): return self.b(x)

class LatentUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(TIMESTEPS,256)
        self.d1 = Block(256)
        self.d2 = Block(256)
        self.u1 = Block(256)
        self.out = nn.Conv2d(256,256,3,1,1)

    def forward(self,z,t):
        e = self.embed(t).view(-1,256,1,1)
        z = z + e
        d1 = self.d1(z)
        d2 = self.d2(d1)
        u1 = self.u1(d2 + d1)
        return self.out(u1)

# ================= PATCHGAN =================
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,4,2,1), nn.LeakyReLU(0.2),
            nn.Conv2d(32,64,4,2,1), nn.LeakyReLU(0.2),
            nn.Conv2d(64,128,4,2,1), nn.LeakyReLU(0.2),
            nn.Conv2d(128,1,4,1,0)
        )
    def forward(self,x): return self.net(x)

# ================= INIT =================
E, G, UNet, D = Encoder().to(device), Decoder().to(device), LatentUNet().to(device), Discriminator().to(device)

opt_G = optim.Adam(list(E.parameters())+list(G.parameters())+list(UNet.parameters()), lr=LR)
opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5,0.999))
bce = nn.BCEWithLogitsLoss()

# ================= DIFFUSION PARAMS =================
betas = torch.linspace(1e-4,0.02,TIMESTEPS).to(device)
alphas = 1 - betas
alpha_hat = torch.cumprod(alphas,0)

def kl_loss(z): return torch.mean(z**2)

# ================= TRAIN =================
print("🚀 Training TRUE Hybrid LDM-GAN")

for epoch in range(300):
    for imgs in tqdm(loader):
        imgs = imgs.to(device)

        # -------- Generator --------
        z = E(imgs)
        recon = G(z)

        ae = F.mse_loss(recon, imgs)
        kl = 0.001*kl_loss(z)

        t = torch.randint(0,TIMESTEPS,(z.size(0),),device=device)
        noise = torch.randn_like(z)
        a = alpha_hat[t].view(-1,1,1,1)

        z_noisy = torch.sqrt(a)*z + torch.sqrt(1-a)*noise
        pred_noise = UNet(z_noisy,t)
        diff = F.mse_loss(pred_noise, noise)

        gan = bce(D(recon), torch.ones_like(D(recon)))

        g_loss = ae + diff + kl + 0.01*gan
        opt_G.zero_grad(); g_loss.backward(); opt_G.step()

        # -------- Discriminator --------
        d_loss = 0.5*(bce(D(imgs),torch.ones_like(D(imgs))) +
                      bce(D(recon.detach()),torch.zeros_like(D(recon))))
        opt_D.zero_grad(); d_loss.backward(); opt_D.step()

    print(f"Epoch {epoch+1} | AE {ae:.3f} Diff {diff:.3f} GAN {gan:.3f}")

    if (epoch+1)%20==0:
        save_image((recon[:8]+1)/2, f"sample_{epoch+1}.png", nrow=4)


In [ ]:
#PL-LDGAN (A HYBRID LDM-GAN TRAINING)
# ============================================================
# # Encoder → 64x64x256 → UNet Diffusion → Decoder → PatchGAN
# AE + KL + Diffusion + GAN
# ============================================================

import os, torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import save_image
from PIL import Image
import torch.nn.functional as F
from tqdm import tqdm

# ================= CONFIG =================
IMG_SIZE = 256
BATCH = 16
TIMESTEPS = 500
LR = 2e-4
DATA_PATH = "/content/drive/MyDrive/POTATODATASET/Healthy"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ================= DATA =================
class LeafDataset(Dataset):
    def __init__(self, root):
        self.paths = [os.path.join(root,f) for f in os.listdir(root)]
        self.tf = T.Compose([
            T.Resize((IMG_SIZE,IMG_SIZE)),
            T.ToTensor(),
            T.Normalize([0.5]*3,[0.5]*3)
        ])
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img)

loader = DataLoader(LeafDataset(DATA_PATH), batch_size=BATCH, shuffle=True)

# ================= ENCODER =================
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,4,2,1), nn.ReLU(),     # 256→128
            nn.Conv2d(64,128,4,2,1), nn.ReLU(),   # 128→64
            nn.Conv2d(128,256,3,1,1), nn.ReLU()   # keep 64
        )
    def forward(self,x): return self.net(x)      # [B,256,64,64]

# ================= DECODER =================
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(256,128,4,2,1), nn.ReLU(), # 64→128
            nn.ConvTranspose2d(128,64,4,2,1), nn.ReLU(),  # 128→256
            nn.Conv2d(64,3,3,1,1),
            nn.Tanh()
        )
    def forward(self,z): return self.net(z)

# ================= UNET DIFFUSION =================
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(c,c,3,1,1), nn.ReLU(),
            nn.Conv2d(c,c,3,1,1), nn.ReLU()
        )
    def forward(self,x): return self.b(x)

class LatentUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(TIMESTEPS,256)
        self.d1 = Block(256)
        self.d2 = Block(256)
        self.u1 = Block(256)
        self.out = nn.Conv2d(256,256,3,1,1)

    def forward(self,z,t):
        e = self.embed(t).view(-1,256,1,1)
        z = z + e
        d1 = self.d1(z)
        d2 = self.d2(d1)
        u1 = self.u1(d2 + d1)
        return self.out(u1)

# ================= PATCHGAN =================
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,4,2,1), nn.LeakyReLU(0.2),
            nn.Conv2d(32,64,4,2,1), nn.LeakyReLU(0.2),
            nn.Conv2d(64,128,4,2,1), nn.LeakyReLU(0.2),
            nn.Conv2d(128,1,4,1,0)
        )
    def forward(self,x): return self.net(x)

# ================= INIT =================
E, G, UNet, D = Encoder().to(device), Decoder().to(device), LatentUNet().to(device), Discriminator().to(device)

opt_G = optim.Adam(list(E.parameters())+list(G.parameters())+list(UNet.parameters()), lr=LR)
opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5,0.999))
bce = nn.BCEWithLogitsLoss()

# ================= DIFFUSION PARAMS =================
betas = torch.linspace(1e-4,0.02,TIMESTEPS).to(device)
alphas = 1 - betas
alpha_hat = torch.cumprod(alphas,0)

def kl_loss(z): return torch.mean(z**2)

# ================= TRAIN =================
print("🚀 Training TRUE Hybrid LDM-GAN")

for epoch in range(300):
    for imgs in tqdm(loader):
        imgs = imgs.to(device)

        # -------- Generator --------
        z = E(imgs)
        recon = G(z)

        ae = F.mse_loss(recon, imgs)
        kl = 0.001*kl_loss(z)

        t = torch.randint(0,TIMESTEPS,(z.size(0),),device=device)
        noise = torch.randn_like(z)
        a = alpha_hat[t].view(-1,1,1,1)

        z_noisy = torch.sqrt(a)*z + torch.sqrt(1-a)*noise
        pred_noise = UNet(z_noisy,t)
        diff = F.mse_loss(pred_noise, noise)

        gan = bce(D(recon), torch.ones_like(D(recon)))

        g_loss = ae + diff + kl + 0.01*gan
        opt_G.zero_grad(); g_loss.backward(); opt_G.step()

        # -------- Discriminator --------
        d_loss = 0.5*(bce(D(imgs),torch.ones_like(D(imgs))) +
                      bce(D(recon.detach()),torch.zeros_like(D(recon))))
        opt_D.zero_grad(); d_loss.backward(); opt_D.step()

    print(f"Epoch {epoch+1} | AE {ae:.3f} Diff {diff:.3f} GAN {gan:.3f}")

    if (epoch+1)%20==0:
        save_image((recon[:8]+1)/2, f"sample_{epoch+1}.png", nrow=4)
